In [ ]:
pip install yt-dlp


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.7 MB/s eta 0:00:00


In [ ]:
pip install -U openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=27f858cf97ca41c76de2fc700554e58b986f93d1ca7a3b44e6e50c3558e2c741
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import os
import json
import subprocess
import re
import whisper
import yt_dlp

GEMINI_API_KEY = ""
YOUTUBE_URL = input("🎥 Paste YouTube Video URL: ")

def get_video_id(url):
    if "v=" in url:
        return url.split("v=")[-1].split("&")[0]
    return url.split("/")[-1]

def get_subtitles(video_url):
    ydl_opts = {
        'writesubtitles': True,
        'writeautomaticsub': True,
        'skip_download': True,
        'subtitleslangs': ['en'],
        'subtitlesformat': 'vtt',
        'outtmpl': '%(id)s.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
        video_id = info.get("id")
        subtitle_file = f"{video_id}.en.vtt"
        if os.path.exists(subtitle_file):
            return subtitle_file
        else:
            return None

def vtt_to_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    content = re.sub(r"\d+\n\d{2}:\d{2}:\d{2}.* --> .*", "", content)
    return re.sub(r"\n{2,}", "\n", content.strip())

def download_audio(video_url, video_id):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{video_id}.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    return f"{video_id}.mp3"

def transcribe_with_whisper(audio_path):
    print("🎙️ Transcribing using Whisper...")
    model = whisper.load_model("base")  # You can also use "small", "medium", "large"
    result = model.transcribe(audio_path)
    return result['text']

def call_gemini_flash(prompt):
    payload = {
        "contents": [{
            "parts": [{"text": prompt}]
        }]
    }

    curl_command = [
        "curl",
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}",
        "-H", "Content-Type: application/json",
        "-X", "POST",
        "-d", json.dumps(payload)
    ]

    result = subprocess.run(curl_command, capture_output=True, text=True)
    try:
        content = json.loads(result.stdout)
        return content['candidates'][0]['content']['parts'][0]['text']
    except:
        return "❌ Failed to get response from Gemini."

def generate_outputs(transcript):
    prompts = {
        "Short Notes": f"Generate short notes from this transcript:\n{transcript}",
        "Detailed Notes": f"Generate detailed notes from this transcript:\n{transcript}",
        "Long Notes": f"Generate long, structured notes from this transcript:\n{transcript}",
        "Question Bank": f"Create a question bank with short, medium, and long answer questions (with answers) from this:\n{transcript}",
        "Previous Year Questions": f"List relevant Indian university previous year questions from this content:\n{transcript}",
        "Practice Sets": f"Create practice questions (Easy, Medium, Hard) from this:\n{transcript}",
        "GATE/NET Questions": f"Create GATE and NET-level competency-based questions:\n{transcript}",
        "Mind Map": f"Generate a hierarchical mind map for this content:\n{transcript}",
        "Research Papers": f"List research papers related to this topic:\n{transcript}",
        "Related Videos": f"Suggest related YouTube videos:\n{transcript}",
        "Book Recommendations": f"Suggest academic books based on this topic:\n{transcript}"
    }

    all_outputs = {}
    for section, prompt in prompts.items():
        print(f"🔍 Generating: {section}")
        response = call_gemini_flash(prompt)
        all_outputs[section] = response
    return all_outputs

def save_to_text(data, filename="YouTube_Generated_Learning.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for section, content in data.items():
            f.write(f"========== {section} ==========\n{content}\n\n")
    print(f"\n✅ All outputs saved to {filename}")

# ----- MAIN PIPELINE -----
video_id = get_video_id(YOUTUBE_URL)
subtitle_file = get_subtitles(YOUTUBE_URL)

if subtitle_file:
    subtitles = vtt_to_text(subtitle_file)
    print("✅ Subtitles extracted!")
    outputs = generate_outputs(subtitles)
    save_to_text(outputs)
else:
    print("⚠️ No subtitles found. Falling back to Whisper transcription...")
    audio_path = download_audio(YOUTUBE_URL, video_id)
    transcript = transcribe_with_whisper(audio_path)
    outputs = generate_outputs(transcript)
    save_to_text(outputs)


[youtube:tab] Extracting URL: https://www.youtube.com/watch?v=kVw59bIbrw0&list=PPSV&t=505s
[youtube:tab] Downloading playlist PPSV - add --no-playlist to download just the video kVw59bIbrw0
[youtube:tab] PPSV: Downloading webpage


[youtube] Extracting URL: https://www.youtube.com/watch?v=kVw59bIbrw0
[youtube] kVw59bIbrw0: Downloading webpage


[youtube] kVw59bIbrw0: Downloading android sdkless player API JSON
[youtube] kVw59bIbrw0: Downloading web safari player API JSON


[youtube] kVw59bIbrw0: Downloading m3u8 information


[info] kVw59bIbrw0: Downloading subtitles: en
⚠️ No subtitles found. Falling back to Whisper transcription...


🎙️ Transcribing using Whisper...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 130MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🔍 Generating: Short Notes
🔍 Generating: Detailed Notes
🔍 Generating: Long Notes
🔍 Generating: Question Bank
🔍 Generating: Previous Year Questions
🔍 Generating: Practice Sets
🔍 Generating: GATE/NET Questions
🔍 Generating: Mind Map
🔍 Generating: Research Papers
🔍 Generating: Related Videos
🔍 Generating: Book Recommendations

✅ All outputs saved to YouTube_Generated_Learning.txt


In [ ]:
import os
import json
import subprocess
import re
import whisper
import yt_dlp

GEMINI_API_KEY = ""
YOUTUBE_URL = input("🎥 Paste YouTube Video URL: ")

def get_video_id(url):
    if "v=" in url:
        return url.split("v=")[-1].split("&")[0]
    return url.split("/")[-1]

def get_subtitles(video_url):
    ydl_opts = {
        'writesubtitles': True,
        'writeautomaticsub': True,
        'skip_download': True,
        'subtitleslangs': ['en'],
        'subtitlesformat': 'vtt',
        'outtmpl': '%(id)s.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)
        video_id = info.get("id")
        subtitle_file = f"{video_id}.en.vtt"
        if os.path.exists(subtitle_file):
            return subtitle_file
        else:
            return None

def vtt_to_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    content = re.sub(r"\d+\n\d{2}:\d{2}:\d{2}.* --> .*", "", content)
    return re.sub(r"\n{2,}", "\n", content.strip())

def download_audio(video_url, video_id):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{video_id}.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    return f"{video_id}.mp3"

def transcribe_with_whisper(audio_path):
    print("🎙️ Transcribing using Whisper...")
    model = whisper.load_model("base")  # You can also use "small", "medium", "large"
    result = model.transcribe(audio_path)
    return result['text']

def call_gemini_flash(prompt):
    payload = {
        "contents": [{
            "parts": [{"text": prompt}]
        }]
    }

    curl_command = [
        "curl",
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}",
        "-H", "Content-Type: application/json",
        "-X", "POST",
        "-d", json.dumps(payload)
    ]

    result = subprocess.run(curl_command, capture_output=True, text=True)
    try:
        content = json.loads(result.stdout)
        return content['candidates'][0]['content']['parts'][0]['text']
    except:
        return "❌ Failed to get response from Gemini."

def generate_outputs(transcript):
    prompts = {
        "Short Notes": f"Generate short notes from this transcript:\n{transcript}",
        "Detailed Notes": f"Generate detailed notes from this transcript:\n{transcript}",
        "Long Notes": f"Generate long, structured notes from this transcript:\n{transcript}",
        "Question Bank": f"Create a question bank with short, medium, and long answer questions (with answers) from this:\n{transcript}",
        "Previous Year Questions": f"List relevant Indian university previous year questions from this content:\n{transcript}",
        "Practice Sets": f"Create practice questions (Easy, Medium, Hard) from this:\n{transcript}",
        "GATE/NET Questions": f"Create GATE and NET-level competency-based questions:\n{transcript}",
        "Mind Map": f"Generate a hierarchical mind map for this content:\n{transcript}",
        "Research Papers": f"List research papers related to this topic:\n{transcript}",
        "Related Videos": f"Suggest related YouTube videos:\n{transcript}",
        "Book Recommendations": f"Suggest academic books based on this topic:\n{transcript}"
    }

    all_outputs = {}
    for section, prompt in prompts.items():
        print(f"🔍 Generating: {section}")
        response = call_gemini_flash(prompt)
        all_outputs[section] = response
    return all_outputs

def save_to_text(data, filename="YouTube_Generated_Learning.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for section, content in data.items():
            f.write(f"========== {section} ==========\n{content}\n\n")
    print(f"\n✅ All outputs saved to {filename}")

# ----- MAIN PIPELINE -----
video_id = get_video_id(YOUTUBE_URL)
subtitle_file = get_subtitles(YOUTUBE_URL)

if subtitle_file:
    subtitles = vtt_to_text(subtitle_file)
    print("✅ Subtitles extracted!")
    outputs = generate_outputs(subtitles)
    save_to_text(outputs)
else:
    print("⚠️ No subtitles found. Falling back to Whisper transcription...")
    audio_path = download_audio(YOUTUBE_URL, video_id)
    transcript = transcribe_with_whisper(audio_path)
    outputs = generate_outputs(transcript)
    save_to_text(outputs)




🎥 Paste YouTube Video URL: https://www.youtube.com/watch?v=1w-2fJAYNmE&list=PPSV
[youtube:tab] Extracting URL: https://www.youtube.com/watch?v=1w-2fJAYNmE&list=PPSV
[youtube:tab] Downloading playlist PPSV - add --no-playlist to download just the video 1w-2fJAYNmE
[youtube:tab] PPSV: Downloading webpage


[youtube] Extracting URL: https://www.youtube.com/watch?v=1w-2fJAYNmE
[youtube] 1w-2fJAYNmE: Downloading webpage
[youtube] 1w-2fJAYNmE: Downloading tv client config
[youtube] 1w-2fJAYNmE: Downloading player 179bab65-main
[youtube] 1w-2fJAYNmE: Downloading tv player API JSON
[youtube] 1w-2fJAYNmE: Downloading ios player API JSON
[youtube] 1w-2fJAYNmE: Downloading m3u8 information
[info] 1w-2fJAYNmE: Downloading subtitles: en
⚠️ No subtitles found. Falling back to Whisper transcription...


🎙️ Transcribing using Whisper...


/usr/local/lib/python3.11/dist-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🔍 Generating: Short Notes
🔍 Generating: Detailed Notes
🔍 Generating: Long Notes
🔍 Generating: Question Bank
🔍 Generating: Previous Year Questions
🔍 Generating: Practice Sets
🔍 Generating: GATE/NET Questions
🔍 Generating: Mind Map
🔍 Generating: Research Papers
🔍 Generating: Related Videos
🔍 Generating: Book Recommendations

✅ All outputs saved to YouTube_Generated_Learning.txt
